# Task 3 — Schema Validation

In [18]:
import pandas as pd
from pathlib import Path

In [19]:
# Define the path to the interim data folder
interim_folder = Path("../data/interim")

# Define the path to the cleaned dataset
cleaned_file = interim_folder / "cleaned.csv"

In [20]:
# Load the cleaned dataset
# Load the cleaned dataset with the expected data types
cleaned_df = pd.read_csv(
    cleaned_file,
    dtype={"symbol": "string"},
    parse_dates=["date"]
)

cleaned_df.head()

,symbol,date,open,high,low,close,adj_close,volume
0,1120,2013-01-01,16.4380,16.5630,16.3130,16.3755,10.5869,7578353
1,1120,2013-01-02,16.3755,16.7505,16.3755,16.6880,10.7890,9374450
2,1120,2013-01-03,61.6173,61.6173,61.6173,61.6173,39.8362,0
3,1120,2013-01-06,16.8755,16.8755,16.8130,16.8130,10.8698,4490083
4,1120,2013-01-07,16.8130,16.8755,16.6255,16.8755,10.9102,11432063


## 1. Define Input Schema

In [21]:
# Define the expected schema
schema = {
    "symbol": {"dtype": "string", "nullable": False},
    "date": {"dtype": "datetime", "nullable": False},
    "open": {"dtype": "float", "nullable": False},
    "high": {"dtype": "float", "nullable": False},
    "low": {"dtype": "float", "nullable": False},
    "close": {"dtype": "float", "nullable": False},
    "adj_close": {"dtype": "float", "nullable": False},
    "volume": {"dtype": "integer", "nullable": False}
}

schema

{'symbol': {'dtype': 'string', 'nullable': False},
 'date': {'dtype': 'datetime', 'nullable': False},
 'open': {'dtype': 'float', 'nullable': False},
 'high': {'dtype': 'float', 'nullable': False},
 'low': {'dtype': 'float', 'nullable': False},
 'close': {'dtype': 'float', 'nullable': False},
 'adj_close': {'dtype': 'float', 'nullable': False},
 'volume': {'dtype': 'integer', 'nullable': False}}

In [22]:
# Check that all expected columns are present
expected_columns = list(schema.keys())

print("Expected columns:", expected_columns)
print("Actual columns:", list(cleaned_df.columns))

Expected columns: ['symbol', 'date', 'open', 'high', 'low', 'close', 'adj_close', 'volume']
Actual columns: ['symbol', 'date', 'open', 'high', 'low', 'close', 'adj_close', 'volume']


In [23]:
# Check the data types
for column, rules in schema.items():
    actual_type = str(cleaned_df[column].dtype)
    print(column, "->", actual_type)

symbol -> string
date -> datetime64[us]
open -> float64
high -> float64
low -> float64
close -> float64
adj_close -> float64
volume -> int64


In [24]:
# Check for missing values
missing_values = cleaned_df.isnull().sum()

missing_values

symbol       0
date         0
open         0
high         0
low          0
close        0
adj_close    0
volume       0
dtype: int64

In [25]:
# Check the stock symbols
cleaned_df["symbol"].unique()

<StringArray>
['1120', '1211', '2010', '2080', '2280', '2310', '3030', '3060', '4003',
 '4030', '4190', '5110', '7010', '7020', '8210']
Length: 15, dtype: string

In [26]:
# Check the date range
print("Start date:", cleaned_df["date"].min())
print("End date:", cleaned_df["date"].max())

Start date: 2013-01-01 00:00:00
End date: 2026-03-31 00:00:00


In [27]:
# Check for invalid price values
price_columns = ["open", "high", "low", "close", "adj_close"]

for column in price_columns:
    print(column, "invalid values:", (cleaned_df[column] <= 0).sum())

open invalid values: 0
high invalid values: 0
low invalid values: 0
close invalid values: 0
adj_close invalid values: 0


In [28]:
# Check for invalid volume values
print("Invalid volume values:", (cleaned_df["volume"] < 0).sum())

Invalid volume values: 0


In [29]:
# Validate the dataset against the schema

allowed_symbols = {
    "1120", "2080", "2010", "2310", "7010",
    "7020", "1211", "2280", "8210", "4003",
    "3060", "3030", "4190", "5110", "4030"
}

valid_rows = (
    cleaned_df["symbol"].isin(allowed_symbols)
    & cleaned_df["symbol"].notna()
    & cleaned_df["date"].notna()
    & cleaned_df["open"].notna()
    & cleaned_df["high"].notna()
    & cleaned_df["low"].notna()
    & cleaned_df["close"].notna()
    & cleaned_df["adj_close"].notna()
    & cleaned_df["volume"].notna()
    & (cleaned_df["open"] > 0)
    & (cleaned_df["high"] > 0)
    & (cleaned_df["low"] > 0)
    & (cleaned_df["close"] > 0)
    & (cleaned_df["adj_close"] > 0)
    & (cleaned_df["volume"] >= 0)
)

print("Valid rows:", valid_rows.sum())
print("Rejected rows:", (~valid_rows).sum())

Valid rows: 49811
Rejected rows: 0


In [30]:
# Separate valid and rejected rows
validated_df = cleaned_df[valid_rows].copy()
rejected_df = cleaned_df[~valid_rows].copy()

print("Validated rows:", len(validated_df))
print("Rejected rows:", len(rejected_df))

Validated rows: 49811
Rejected rows: 0


In [31]:
# Save the validated dataset
validated_df.to_csv(
    interim_folder / "validated.csv",
    index=False
)

print("Saved:", interim_folder / "validated.csv")

Saved: ..\data\interim\validated.csv


In [32]:
# Save rejected rows
rejected_df.to_csv(
    interim_folder / "rejected.csv",
    index=False
)

print("Saved:", interim_folder / "rejected.csv")

Saved: ..\data\interim\rejected.csv


In [33]:
# Check the validation results
print("Validated rows:", len(validated_df))
print("Rejected rows:", len(rejected_df))

Validated rows: 49811
Rejected rows: 0
